01-lib.ipynb
02-corpus.ipynb
03-vocab.ipynb
04-dtm-tfidf.ipynb
05-pca.ipynb
06-lda.ipynb
07-sentiment.ipynb
08-word2vec.ipynb
09-riffs.ipynb

## Construct CORPUS table

First I am going to look at the files that make up each of the StandardEbooks.


For Giant's Bread which has book and chapter separations, I am going to do the following: as book in the giants bread is a product of publishing (i believe) rather than narrative, I will just continue the numbering making 2-1 be the next number after 1-x ends (REWRITE)

In [126]:
# import libraries
import pandas as pd
import glob
import os
import re
import nltk
import xml.etree.ElementTree as ET
from pathlib import Path
from bs4 import BeautifulSoup

In [127]:
nltk_resources = [
    'punkt_tab',
    'averaged_perceptron_tagger_eng',
    'stopwords',
    'tagsets'
]

for resource in nltk_resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Could not download {resource}: {e}")

In [128]:
# get list of all files in the raw_data folder
root_dir = 'raw_data'
file_dict = {}

# get paths to xhtml files
paths = sorted(Path(root_dir).glob('*/src/epub/text/*.xhtml')) # glob finds all the pathnames matching a specified pattern

for path in paths:
    book_id = path.parts[1].replace('agatha-christie_', '') # .parts breaks a path into a tuple of components
    # for each book_id, get the file path to all xhtml files
    if book_id not in file_dict:
        file_dict[book_id] = []
    file_dict[book_id].append(path)

file_dict

{'giants-bread': [PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-1.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-2.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-3.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-4.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/book-5.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-1.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-2.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-3.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-4.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-5.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/text/chapter-1-6.xhtml'),
  PosixPath('raw_data/agatha-christie_giants-bread/src/epub/tex

### Write helper functions

In [129]:
BOILERPLATE = {'colophon', 'imprint', 'titlepage', 'uncopyright', 
               'halftitlepage', 'loi', 'dedication'}

def is_content(path):
    stem = path.stem
    if stem in BOILERPLATE:
        return False
    if stem.startswith('book-'):
        return False
    return True

In [130]:
GIANTS_BREAD_CHAPTER_COUNTS = {1: 8, 2: 7, 3: 4, 4: 4, 5: 5}

# make helper function
def get_chap_num(book_id, stem):
    if stem == 'prologue':
        chap_num = 0
    elif book_id == 'giants-bread' and stem.startswith('chapter-'):
        # get the numbers
        N = int(stem.split('-')[1])
        M = int(stem.split('-')[2])
        # cumulative offset for giants bread chapters
        chap_num = sum(count for book, count in GIANTS_BREAD_CHAPTER_COUNTS.items() if book < N) + M
    elif stem.startswith('chapter-'):
        # get the number
        N = int(stem.split('-')[1])
        chap_num = N
    else:
        raise ValueError(f"Cannot determine chap_num for book_id='{book_id}', stem='{stem}'")
    
    return chap_num

In [131]:
# test
print(get_chap_num('the-big-four', 'chapter-5'))        # expect 5
print(get_chap_num('giants-bread', 'chapter-2-3'))      # expect 11
print(get_chap_num('the-secret-adversary', 'prologue')) # expect 0

5
11
0


In [132]:
def tokenize_file(book_id, chap_num, xhtml_path):
    """
    Returns a TOKENS dataframe for one file.
    Uses some of the logic from the tokenize_source() function from HW 4.
    """
    # open file and parse with BeautifulSoup
    with open(xhtml_path, "r") as f:
        soup = BeautifulSoup(f, 'xml')
    # find all <p> tags to get list of paragraph text and filter out chap title
    text_paras = [p.get_text() for p in soup.find_all('p')
                  if 'title' not in p.get('epub:type', '')] # this is a list

    # create paragraph dataframe (must wrap the list in a series to use to_frame)
    PARAS = pd.Series(text_paras).to_frame('para_str')

    # parse into sentences with NLTK's sent_tokenize() method
    SENTS = PARAS.para_str.apply(lambda x: pd.Series(nltk.sent_tokenize(x), dtype = 'string'))\
        .stack().to_frame('sent_str')
    # cleanup (don't need PARAS df object anymore)
    del(PARAS)
    # drop NaN values from sents (empty paragraphs that became NaN after stack operation)
    SENTS = SENTS.dropna(subset=['sent_str'])

    # parse into tokens with NLTK's word_tokenize() method
    TOKENS = SENTS.sent_str.apply(lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x))))\
        .stack().to_frame('pos_tuple')
    # drop NaN values (empty tuples that became NaN after stack operation)
    TOKENS = TOKENS.dropna(subset=['pos_tuple'])
    TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
    TOKENS['pos_group'] = TOKENS.pos.str[:2]
    TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
    # cleanup: don't need pos_tuple anymore
    TOKENS = TOKENS.drop('pos_tuple', axis = 1)
    TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"[\W_]+", "", regex = True)
    TOKENS = TOKENS[TOKENS.term_str != ''].copy() # copy() ensures working on a dataframe rather than a view of one; good to include after bool filtering
    # cleanup (don't need SENTS df object anymore)
    del(SENTS)

    # add the chap_num and book_id (which are constant per file)
    TOKENS['chap_num'] = chap_num
    TOKENS['book_id'] = book_id

    # name, set, and order indices
    TOKENS.index.names = ['para_num', 'sent_num', 'token_num']
    TOKENS = TOKENS.set_index(['book_id', 'chap_num'], append=True)
    TOKENS = TOKENS.reorder_levels(['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num'])

    return TOKENS

In [133]:
# test
tokenize_file('the-big-four', 1, 'raw_data/agatha-christie_the-big-four/src/epub/text/chapter-1.xhtml')

pos pos_group token_str  \
book_id      chap_num para_num sent_num token_num                            
the-big-four 1        0        0        0          PRP        PR         I   
                                        1          VBP        VB      have   
                                        2          VBN        VB       met   
                                        3          NNS        NN    people   
                                        4           WP        WP       who   
...                                                ...       ...       ...   
                      100      1        11         PRP        PR         I   
                                        12          MD        MD       can   
                                        13          VB        VB    manage   
                                        14          DT        DT       the   
                                        15          NN        NN      rest   

                                                  term_str  
book_id      chap_num para_num sent_num token_num           
the-big-four 1        0        0        0                i  
                                        1             have  
                                        2              met  
                                        3           people  
                                        4              who  
...                                                    ...  
                      100      1        11               i  
                                        12             can  
                                        13          manage  
                                        14             the  
                                        15            rest  

[2780 rows x 4 columns]

### Main parsing loop

In [ ]:
# tokenize source and append to list of corpora

# initialize list of corpora
corpus = []

# do i need a comment here???
for book_id, paths in file_dict.items():
    for path in paths:
        if is_content(path): # if not boilerplate
            print(book_id, path.stem) # see which file is being processed
            # if poirot-investigates, I to handle the short stories (stored the way all other books have chapters) as books themselves
            if book_id == 'poirot-investigates': # this is the book_id from the filepath, not the book_id I use in LIB
                story_id = path.stem
                chap_num = 1
                corpus.append(tokenize_file(story_id, chap_num, path))
            # if any other work
            else:
                chap_num = get_chap_num(book_id, path.stem)
                corpus.append(tokenize_file(book_id, chap_num, path))

CORPUS = pd.concat(corpus) # the dataframes already have the full OHCO index set (do not set here or we end up with another level)
print("\nDone!")
# cleanup
del(corpus)

giants-bread chapter-1-1
giants-bread chapter-1-2
giants-bread chapter-1-3
giants-bread chapter-1-4
giants-bread chapter-1-5
giants-bread chapter-1-6
giants-bread chapter-1-7
giants-bread chapter-1-8
giants-bread chapter-2-1
giants-bread chapter-2-2
giants-bread chapter-2-3
giants-bread chapter-2-4
giants-bread chapter-2-5
giants-bread chapter-2-6
giants-bread chapter-2-7
giants-bread chapter-3-1
giants-bread chapter-3-2
giants-bread chapter-3-3
giants-bread chapter-3-4
giants-bread chapter-4-1
giants-bread chapter-4-2
giants-bread chapter-4-3
giants-bread chapter-4-4
giants-bread chapter-5-1
giants-bread chapter-5-2
giants-bread chapter-5-3
giants-bread chapter-5-4
giants-bread chapter-5-5
giants-bread prologue
poirot-investigates the-adventure-of-the-cheap-flat
poirot-investigates the-adventure-of-the-egyptian-tomb
poirot-investigates the-adventure-of-the-italian-nobleman
poirot-investigates the-adventure-of-the-western-star
poirot-investigates the-case-of-the-missing-will
poirot-inv

In [138]:
CORPUS

pos pos_group  \
book_id                 chap_num para_num sent_num token_num                  
giants-bread            1        0        0        0           EX        EX   
                                                   1          VBD        VB   
                                                   2           RB        RB   
                                                   3           CD        CD   
                                                   4          NNS        NN   
...                                                           ...       ...   
the-seven-dials-mystery 9        117      6        4           IN        IN   
                                                   5          NNP        NN   
                                                   6          NNP        NN   
                                                   7           RB        RB   
                                                   8           JJ        JJ   

                                                               token_str  \
book_id                 chap_num para_num sent_num token_num               
giants-bread            1        0        0        0               There   
                                                   1                were   
                                                   2                only   
                                                   3               three   
                                                   4              people   
...                                                                  ...   
the-seven-dials-mystery 9        117      6        4                  of   
                                                   5             Loraine   
                                                   6                Wade   
                                                   7              highly   
                                                   8          suspicious   

                                                                term_str  
book_id                 chap_num para_num sent_num token_num              
giants-bread            1        0        0        0               there  
                                                   1                were  
                                                   2                only  
                                                   3               three  
                                                   4              people  
...                                                                  ...  
the-seven-dials-mystery 9        117      6        4                  of  
                                                   5             loraine  
                                                   6                wade  
                                                   7              highly  
                                                   8          suspicious  

[852006 rows x 4 columns]